# stochastic-rs — CUDA back-end check on a Colab GPU

Runs the CUDA tests of `stochastic-rs-stochastic` on the notebook's GPU, so every CUDA route can be validated without a CUDA machine of your own: the hand-written `cuda` back-end (cuFFT + Philox for fGN, NVRTC kernels for the Euler engine), every process the engine serves — the diffusions, the stochastic-volatility systems, the curve-driven short-rate models, the jump models, the curve-driven term structures, and the fractional ones, which feed the kernels from the fGN pipeline without a host round trip — including the correlated fractional pair, whose two rows come out of one embedding.

Before running: **Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works). The first cargo build takes 10–20 minutes on Colab's two cores; the tests themselves take seconds.

Cells: 1 GPU + toolkit check · 2 Rust toolchain · 3 clone · 4 native CUDA tests · 5 every process the engine serves, on the GPU · 6 chunk invariance · 6b the determinism gate under load · 7 engine throughput and kernel occupancy · 8 (optional) Python wheel with `device="cuda"` · 9 (optional) plots from the GPU.

In [ ]:
# 1. The GPU and toolkit: the driver's CUDA version (nvidia-smi) must be >= nvcc's, or it cannot JIT
#    NVRTC's PTX (CUDA_ERROR_UNSUPPORTED_PTX_VERSION); Colab matches them, else see the end.
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
# 2. Rust (stable, minimal profile). PATH is extended for every later cell.
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CARGO_TERM_COLOR"] = "never"
!cargo --version && rustc --version

In [ ]:
# 3. The repository. REF is a branch, tag or commit; main is the default.
REF = "main"
!rm -rf stochastic-rs && git clone --quiet https://github.com/rust-dd/stochastic-rs.git && git -C stochastic-rs checkout --quiet {REF}
%cd stochastic-rs
!git log --oneline -1

In [ ]:
# 4. Native CUDA tests, ~10-20 min to build: fGN (cuFFT + Philox; shapes, moments, seeds whatever
#    the launch history) and the Euler engine (GBM moments, CIR positivity, f32/f64, chunking).
!cargo test -p stochastic-rs-stochastic --features cuda --lib -- cuda 2>&1 | grep -E "^test |^test result|error|panicked"

In [ ]:
# 5. Every process the Euler engine serves, on the GPU: the device-law suite samples each on the CPU
#    and on CUDA and compares a terminal statistic plus every boundary the process promises.
!cargo test -p stochastic-rs-stochastic --features cuda --test device_law 2>&1 | grep -E "^test |^test result|error|panicked"


In [ ]:
# 6. Chunk invariance: a batch past the device budget runs in chunks that must equal one launch bit
#    for bit, which a fractional family's pipeline counter could break unseen by any statistic.
!cargo test -p stochastic-rs-stochastic --features cuda --lib chunks_are_bit_identical 2>&1 | grep -E "^test |^test result|error|panicked"


In [ ]:
# 6b. The determinism gate, the one cell whose failure is the finding: eight threads resample one
#     seed under contention against a reference, and any mismatch breaks the bit-identity promise.
REPEATS = 8

import subprocess, time

failures = 0
for i in range(REPEATS):
    start = time.time()
    r = subprocess.run(
        ["cargo", "test", "-p", "stochastic-rs-stochastic",
         "--test", "determinism_under_contention"],
        capture_output=True, text=True,
    )
    ok = "test result: ok" in r.stdout
    failures += 0 if ok else 1
    print(f"  round {i + 1}/{REPEATS}  {'ok' if ok else 'FAILED'}  "
          f"{time.time() - start:5.1f}s", flush=True)
    if not ok:
        for line in (r.stdout + r.stderr).splitlines():
            if "not the seed" in line or "panicked" in line or "FAILED" in line:
                print("     ", line[:200])

print(f"\n{failures} of {REPEATS} rounds were not reproducible")

In [ ]:
# 7. Is the GPU worth it for the Euler engine? The kernel as the driver sees it, then the throughput
#    against this CPU and the fGN pipeline for contrast, each the best of five runs.
!cargo run --release --example cuda_engine_profile --features cuda 2>&1 | grep -vE "^(warning|note|   Compiling|    Finished|     Running)"

In [ ]:
# 8. (Optional) the CUDA wheel, 20-30 min on Colab, and its device pytest cases; maturin takes an
#    explicit interpreter as Colab has no venv. RUN_PYTHON = False skips it.
RUN_PYTHON = True
WHEEL_FEATURES = "cuda"
if RUN_PYTHON:
    !pip -q install maturin pytest numpy
    # No virtualenv on Colab, so build the wheel for the notebook's interpreter and pip-install it.
    !maturin build --release --features {WHEEL_FEATURES} --interpreter python3 --out dist 2>&1 | tail -2
    !pip -q install --force-reinstall --no-deps dist/*.whl
    !python -m pytest -q stochastic-rs-py/tests/test_stochastic.py -k "device or probe" 2>&1 | tail -8
    import importlib
    import numpy as np
    sr = importlib.import_module("stochastic_rs")
    names = ["cuda", "cuda:0"]
    for name in names:
        print(f"probe_device({name!r}) ->", sr.probe_device(name))
    # Every device name is one concrete backend: cpu, accelerate, cuda, metal.
    # There is no name meaning "whatever is compiled".
    try:
        sr.probe_device("gpu")
    except ValueError as e:
        print('probe_device("gpu") ->', e)
    paths = sr.PyGbm(0.05, 0.2, 253, x0=100.0, t=1.0, seed=7, device="cuda").sample_par(20_000)
    print(paths.shape, paths.dtype, "terminal mean / forward =", paths[:, -1].mean() / (100.0 * np.exp(0.05)))

    # A fractional process: its fGN increments meet the Euler kernel in device memory, and the two
    # runs draw their own streams, so only the laws must agree.
    import time
    def timed(**kw):
        t = time.perf_counter()
        p = sr.PyFou(0.7, 2.0, 1.0, 0.3, 512, x0=0.0, t=1.0, seed=9, **kw).sample_par(2_000)
        return (time.perf_counter() - t) * 1e3, p
    _ = timed(dtype="f32", device="cuda")
    gpu_ms, gpu = timed(dtype="f32", device="cuda")
    cpu_ms, cpu = timed()
    print(f"fOU 2000x512  cpu {cpu_ms:.1f} ms  cuda {gpu_ms:.1f} ms")
    print(f"terminal mean cpu {cpu[:, -1].mean():.4f}  cuda {gpu[:, -1].mean():.4f}")


In [ ]:
# 9. Plots from the GPU (needs cell 8's wheel): fBM paths, the t^(2H) variance law GPU vs CPU, the
#    GBM terminal law against the lognormal density, and wall time per call CPU vs CUDA.
if RUN_PYTHON:
    import importlib, time
    import numpy as np
    import matplotlib.pyplot as plt
    sr = importlib.import_module("stochastic_rs")
    print(sr.probe_device("cuda"))
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # (a) fBM paths for two Hurst exponents, sampled on the GPU (f64).
    ax = axes[0, 0]
    n = 1024
    grid = np.linspace(0.0, 1.0, n)
    for h, color in ((0.3, "tab:red"), (0.8, "tab:blue")):
        paths = sr.PyFbm(h, n, t=1.0, seed=3, device="cuda").sample_par(4)
        for k, path in enumerate(paths):
            ax.plot(grid, path, color=color, lw=0.8, alpha=0.85, label=f"H = {h}" if k == 0 else None)
    ax.set_title("fBM paths, Cuda f64")
    ax.set_xlabel("t")
    ax.legend()

    # (b) Var B_H(t) = t^(2H): GPU dots, CPU crosses, theory lines.
    ax = axes[0, 1]
    m = 4000
    keep = grid > 0.05
    for h, color in ((0.3, "tab:red"), (0.5, "tab:green"), (0.8, "tab:blue")):
        gpu = sr.PyFbm(h, n, t=1.0, seed=11, device="cuda").sample_par(m).var(axis=0)
        cpu = sr.PyFbm(h, n, t=1.0, seed=11, device="cpu").sample_par(m).var(axis=0)
        ax.loglog(grid[keep], gpu[keep], ".", color=color, ms=3, label=f"GPU H={h}")
        ax.loglog(grid[keep], cpu[keep], "x", color=color, ms=3, alpha=0.5, label=f"CPU H={h}")
        ax.loglog(grid[keep], grid[keep] ** (2 * h), "k-", lw=0.8)
    ax.set_title("Var B_H(t) against t^(2H) (black lines), 4000 paths")
    ax.set_xlabel("t")
    ax.legend(fontsize=7, ncol=2)

    # (c) GBM terminal value from the Euler engine against the lognormal density.
    ax = axes[1, 0]
    mu, sigma, s0, horizon = 0.05, 0.2, 100.0, 1.0
    terminal = sr.PyGbm(mu, sigma, 253, x0=s0, t=horizon, seed=7, device="cuda").sample_par(100_000)[:, -1]
    ax.hist(terminal, bins=120, density=True, alpha=0.6, label="Cuda Euler, 100k paths")
    x = np.linspace(terminal.min(), terminal.max(), 400)
    mlog = np.log(s0) + (mu - 0.5 * sigma**2) * horizon
    slog = sigma * np.sqrt(horizon)
    ax.plot(x, np.exp(-(np.log(x) - mlog) ** 2 / (2 * slog**2)) / (x * slog * np.sqrt(2 * np.pi)), "k-", label="lognormal density")
    ax.set_title(f"GBM S_T: sample mean {terminal.mean():.3f}, theory {s0 * np.exp(mu * horizon):.3f}")
    ax.legend()

    # (d) wall time per call, CPU (Colab's two cores) against the T4; warm calls.
    ax = axes[1, 1]
    def timed(fn, warm=1, reps=3):
        for _ in range(warm):
            fn()
        start = time.perf_counter()
        for _ in range(reps):
            fn()
        return (time.perf_counter() - start) / reps
    cases = {
        "fGN n=4096\nm=2000": lambda d: sr.PyFgn(0.7, 4096, t=1.0, seed=1, device=d).sample_par(2000),
        "fGN n=16384\nm=1000": lambda d: sr.PyFgn(0.7, 16384, t=1.0, seed=1, device=d).sample_par(1000),
        "GBM Euler n=253\nm=100k": lambda d: sr.PyGbm(0.05, 0.2, 253, x0=100.0, t=1.0, seed=1, device=d).sample_par(100_000),
    }
    labels, cpu_s, gpu_s = [], [], []
    for name, fn in cases.items():
        labels.append(name)
        cpu_s.append(timed(lambda: fn("cpu")))
        gpu_s.append(timed(lambda: fn("cuda")))
    xs = np.arange(len(labels))
    ax.bar(xs - 0.2, cpu_s, 0.4, label="CPU (Colab, 2 vCPU)")
    ax.bar(xs + 0.2, gpu_s, 0.4, label="Cuda (T4)")
    for i, (c, g) in enumerate(zip(cpu_s, gpu_s)):
        ax.text(i, max(c, g) * 1.15, f"{c / g:.1f}x", ha="center", fontsize=9)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yscale("log")
    ax.set_ylabel("seconds per call (mean of 3, warm)")
    ax.set_title("Wall time per sample_par: CPU vs CUDA")
    ax.legend()
    plt.tight_layout()
    plt.show()


## Reading the result

Every `cuda*` test prints `ok` and the summary line ends in `0 failed`, and cell 5's device-law suite does the same for every process the engine serves. `probe_device("cuda")` reports `backend: 'Cuda'` — the name says which route ran. Two things are worth a look if something is red:

- `CUDA_ERROR_UNSUPPORTED_PTX_VERSION` on module load: the toolkit (nvcc) is newer than the driver. Either pick a runtime whose versions match, or install the toolkit matching `nvidia-smi`'s CUDA version (`apt-get install cuda-toolkit-12-x`) and rerun cell 4.
- `device unavailable: CudaContext: ...`: no GPU in this runtime. Change the runtime type to a GPU and run from cell 1.

The same commands run on any CUDA machine. With several GPUs, name one on the handle (`with_backend(Cuda::new(n))`, or `Cuda::from_env()` for `STOCHASTIC_RS_DEVICE=n`) or from Python (`device="cuda:n"`, or `STOCHASTIC_RS_DEVICE=n` for a bare `"cuda"`).